# Phase 4 — Dataset Preparation

**Project:** Banking Transaction Monitoring & Fraud Analytics Platform
**Database:** `federal_bank`

This notebook fills the database created in Phase 3 with realistic **synthetic**
data (using the `Faker` library) and produces the transaction feed used later.

**What it does**
1. Connects to MySQL and clears any old data (so it can be re-run).
2. Generates and inserts: branches, employees, customers, accounts, cards, loans.
3. Generates realistic **transactions** and saves them to `datasets/transactions.csv`
   (the Phase 5 simulator streams this file into the `transactions` table).
4. **Seeds deliberate fraud scenarios** and labels them in `datasets/fraud_ground_truth.csv`
   so we can measure detection accuracy in Phase 7.

> All data is fake. No real customer information is used.

## 1. Configuration
Change these values to control the connection and dataset size.

In [ ]:
# --- MySQL connection settings (EDIT the password to match your MySQL) ---
DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": "YOUR_MYSQL_PASSWORD",   # <-- change this to your MySQL root password
    "database": "federal_bank",          # the database created in Phase 3
}

# --- How much data to generate ---
RANDOM_SEED    = 42       # fixed seed => identical data every run (reproducible)
N_BRANCHES     = 8        # number of bank branches
N_EMPLOYEES    = 40       # number of employees across branches
N_CUSTOMERS    = 500      # number of customers
N_TRANSACTIONS = 20000    # number of NORMAL transactions to generate

# --- Where to save the output files ---
DATASETS_DIR   = "datasets"
TX_CSV_PATH    = "datasets/transactions.csv"         # main feed (streamed in Phase 5)
TRUTH_CSV_PATH = "datasets/fraud_ground_truth.csv"   # labels of seeded fraud

## 2. Imports and random seeds
Seeding every random source makes the generated data reproducible.

In [ ]:
import os
import random
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from faker import Faker
import mysql.connector   # install with: pip install mysql-connector-python

Faker.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

fake = Faker("en_IN")               # Indian-locale names and phone numbers
rng  = random.Random(RANDOM_SEED)   # our own random generator

# Historical transactions span roughly the last 9 months.
DATA_END   = datetime.now().replace(microsecond=0)
DATA_START = DATA_END - timedelta(days=270)

os.makedirs(DATASETS_DIR, exist_ok=True)
print("Setup complete. Data spans", DATA_START, "to", DATA_END)

## 3. Connect to MySQL

In [ ]:
conn = mysql.connector.connect(**DB_CONFIG)
cursor = conn.cursor()
print("Connected to database:", DB_CONFIG["database"])

## 4. Clear existing data
Foreign-key checks are disabled, every table is emptied, then checks are re-enabled.
`TRUNCATE` is fast and resets auto-increment ids to 1. This makes the notebook re-runnable.

In [ ]:
cursor.execute("SET FOREIGN_KEY_CHECKS = 0")
for table in ["fraud_alerts", "transactions", "loans", "cards",
              "accounts", "customers", "employees", "branches"]:
    cursor.execute(f"TRUNCATE TABLE {table}")
cursor.execute("SET FOREIGN_KEY_CHECKS = 1")
conn.commit()
print("All tables cleared.")

## 5. Reference data: Indian cities
A spread-out city list lets us build realistic "impossible travel" fraud later.

In [ ]:
CITIES = [
    ("Mumbai", "Maharashtra"),
    ("Delhi", "Delhi"),
    ("Bengaluru", "Karnataka"),
    ("Chennai", "Tamil Nadu"),
    ("Kolkata", "West Bengal"),
    ("Hyderabad", "Telangana"),
    ("Pune", "Maharashtra"),
    ("Ahmedabad", "Gujarat"),
    ("Jaipur", "Rajasthan"),
    ("Lucknow", "Uttar Pradesh"),
]
CITY_NAMES = [c for c, _ in CITIES]
print(len(CITIES), "cities available.")

## 6. Branches
We insert branches, then read their auto-generated ids back to use as foreign keys.

In [ ]:
branch_data = []
for i in range(N_BRANCHES):
    city, state = CITIES[i % len(CITIES)]
    branch_data.append((
        f"Federal Bank - {city}",     # branch_name
        f"FDRL0{i:06d}",              # ifsc_code (11 chars, unique)
        city, state,
        fake.date_between(start_date="-15y", end_date="-2y"),  # opened_date
    ))

cursor.executemany(
    "INSERT INTO branches (branch_name, ifsc_code, city, state, opened_date) "
    "VALUES (%s, %s, %s, %s, %s)",
    branch_data,
)
conn.commit()

cursor.execute("SELECT branch_id FROM branches")
branch_ids = [row[0] for row in cursor.fetchall()]
print(f"Inserted {len(branch_ids)} branches.")

## 7. Employees
Roles are cycled so every branch has fraud analysts (who review alerts in later phases).

In [ ]:
ROLES = ["Teller", "Branch Manager", "Fraud Analyst", "Operations"]
employee_data = []
for i in range(N_EMPLOYEES):
    first, last = fake.first_name(), fake.last_name()
    role = ROLES[i % len(ROLES)]
    email = f"{first}.{last}.{i}@federalbank.example".lower()
    employee_data.append((
        rng.choice(branch_ids), first, last, role, email,
        fake.date_between(start_date="-8y", end_date="today"),
    ))

cursor.executemany(
    "INSERT INTO employees (branch_id, first_name, last_name, role, email, hire_date) "
    "VALUES (%s, %s, %s, %s, %s, %s)",
    employee_data,
)
conn.commit()
print(f"Inserted {len(employee_data)} employees.")

## 8. Customers

In [ ]:
GENDERS = ["Male", "Female", "Other"]
KYC = ["Verified", "Pending", "Rejected"]
customer_data = []
for i in range(N_CUSTOMERS):
    gender = rng.choices(GENDERS, weights=[0.48, 0.48, 0.04])[0]
    first = fake.first_name_male() if gender == "Male" else fake.first_name_female()
    last = fake.last_name()
    city, state = rng.choice(CITIES)
    customer_data.append((
        first, last, gender,
        fake.date_of_birth(minimum_age=18, maximum_age=80),
        f"{first}.{last}.{i}@example.com".lower(),     # unique email
        fake.msisdn()[:10],                            # 10-digit phone
        city, state,
        rng.choices(KYC, weights=[0.85, 0.12, 0.03])[0],
        fake.date_between(start_date="-10y", end_date="-30d"),  # customer_since
    ))

cursor.executemany(
    "INSERT INTO customers "
    "(first_name, last_name, gender, date_of_birth, email, phone, city, state, kyc_status, customer_since) "
    "VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)",
    customer_data,
)
conn.commit()
cursor.execute("SELECT customer_id FROM customers")
customer_ids = [r[0] for r in cursor.fetchall()]
print(f"Inserted {len(customer_ids)} customers.")

## 9. Accounts
Each customer gets 1-2 accounts. Balance ranges depend on account type.

In [ ]:
ACCOUNT_TYPES = ["Savings", "Current", "Fixed Deposit"]
account_data = []
for cust_id in customer_ids:
    for _ in range(rng.choices([1, 2], weights=[0.7, 0.3])[0]):
        acc_type = rng.choices(ACCOUNT_TYPES, weights=[0.6, 0.25, 0.15])[0]
        if acc_type == "Fixed Deposit":
            balance = round(rng.uniform(50000, 1000000), 2)
        elif acc_type == "Current":
            balance = round(rng.uniform(5000, 500000), 2)
        else:
            balance = round(rng.uniform(500, 200000), 2)
        account_data.append((
            cust_id, rng.choice(branch_ids),
            f"{rng.randint(10**11, 10**12 - 1)}",   # 12-digit account number
            acc_type, balance,
            rng.choices(["Active", "Dormant", "Closed"], weights=[0.9, 0.08, 0.02])[0],
            fake.date_between(start_date="-8y", end_date="today"),
        ))

cursor.executemany(
    "INSERT INTO accounts "
    "(customer_id, branch_id, account_number, account_type, balance, status, opened_date) "
    "VALUES (%s, %s, %s, %s, %s, %s, %s)",
    account_data,
)
conn.commit()

# Read accounts back WITH the owner's city (used for transaction locations).
cursor.execute(
    "SELECT a.account_id, a.customer_id, c.city "
    "FROM accounts a JOIN customers c ON a.customer_id = c.customer_id"
)
account_rows = cursor.fetchall()          # [(account_id, customer_id, city), ...]
account_ids = [r[0] for r in account_rows]
print(f"Inserted {len(account_ids)} accounts.")

## 10. Cards
About 70% of accounts get a card. `card_number` is synthetic — never a real number.

In [ ]:
card_data = []
for acc_id in account_ids:
    if rng.random() < 0.7:
        issued = fake.date_between(start_date="-5y", end_date="today")
        card_data.append((
            acc_id,
            f"{rng.randint(10**15, 10**16 - 1)}",   # 16-digit synthetic number
            rng.choices(["Debit", "Credit"], weights=[0.8, 0.2])[0],
            rng.choice(["Visa", "Mastercard", "RuPay"]),
            issued, issued + timedelta(days=5 * 365),
            rng.choices(["Active", "Blocked", "Expired"], weights=[0.92, 0.05, 0.03])[0],
        ))

cursor.executemany(
    "INSERT INTO cards "
    "(account_id, card_number, card_type, network, issued_date, expiry_date, status) "
    "VALUES (%s, %s, %s, %s, %s, %s, %s)",
    card_data,
)
conn.commit()

# Map each account to a card_id (or None).
cursor.execute("SELECT account_id, card_id FROM cards")
account_cards = {}
for acc_id, card_id in cursor.fetchall():
    account_cards.setdefault(acc_id, card_id)
for acc_id in account_ids:
    account_cards.setdefault(acc_id, None)
carded_accounts = [a for a in account_ids if account_cards[a] is not None]
print(f"Inserted {len(card_data)} cards.")

## 11. Loans
About 20% of customers have a loan.

In [ ]:
LOAN_TYPES = ["Home", "Personal", "Auto", "Education"]
loan_data = []
for cust_id in customer_ids:
    if rng.random() < 0.2:
        principal = round(rng.uniform(50000, 5000000), 2)
        loan_data.append((
            cust_id, rng.choice(branch_ids), rng.choice(LOAN_TYPES),
            principal, round(rng.uniform(7.5, 14.0), 2),
            rng.choice([12, 24, 36, 60, 120, 240]),
            round(principal * rng.uniform(0.2, 1.0), 2),
            fake.date_between(start_date="-6y", end_date="today"),
            rng.choices(["Active", "Closed", "Defaulted"], weights=[0.8, 0.15, 0.05])[0],
        ))

cursor.executemany(
    "INSERT INTO loans "
    "(customer_id, branch_id, loan_type, principal_amount, interest_rate, tenure_months, "
    " outstanding_balance, start_date, status) "
    "VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)",
    loan_data,
)
conn.commit()
print(f"Inserted {len(loan_data)} loans.")

## 12. Helper maps for transactions
`city_of_account` = where an account normally transacts.
`account_profile` = an account's typical spend (the amount-anomaly rule looks for values far above this).

In [ ]:
city_of_account = {acc_id: city for (acc_id, cust_id, city) in account_rows}
account_profile = {acc_id: round(rng.uniform(800, 8000), 2) for acc_id in account_ids}
print("Helper maps ready for", len(account_ids), "accounts.")

## 13. Transaction generation functions
Written as small functions so the logic is clear and testable.

In [ ]:
CHANNELS  = ["ATM", "POS", "Internet", "Mobile", "Branch", "UPI"]
CHANNEL_W = [0.15, 0.20, 0.12, 0.20, 0.08, 0.25]
TX_TYPES  = ["Deposit", "Withdrawal", "Transfer", "Payment", "Reversal"]
TX_TYPE_W = [0.15, 0.25, 0.20, 0.35, 0.05]
STATUSES  = ["Success", "Failed", "Reversed"]
STATUS_W  = [0.92, 0.06, 0.02]

# Column order for the CSV — must match what the Phase 5 simulator inserts.
TX_COLUMNS = [
    "account_id", "card_id", "transaction_type", "channel", "amount", "currency",
    "status", "transaction_city", "counterparty_account", "description",
    "transaction_timestamp",
]


def _amount_around(mean):
    # A positive amount that varies around an account's typical spend.
    return round(abs(rng.gauss(mean, mean * 0.4)) + 10, 2)


def _random_timestamp():
    # A random datetime within the historical window.
    secs = rng.randint(0, int((DATA_END - DATA_START).total_seconds()))
    return DATA_START + timedelta(seconds=secs)


def _fmt(ts):
    # Format a datetime the way MySQL DATETIME expects.
    return ts.strftime("%Y-%m-%d %H:%M:%S")


def make_normal_transaction(acc_id):
    # Build ONE ordinary transaction (a dict keyed by column name).
    channel = rng.choices(CHANNELS, weights=CHANNEL_W)[0]
    tx_type = rng.choices(TX_TYPES, weights=TX_TYPE_W)[0]
    status  = rng.choices(STATUSES, weights=STATUS_W)[0]
    # A card is attached only for card-based channels, and only if the account has one.
    card_id = account_cards.get(acc_id) if channel in ("POS", "ATM") else None
    counterparty = f"{rng.randint(10**11, 10**12 - 1)}" if tx_type == "Transfer" else None
    return {
        "account_id": acc_id,
        "card_id": card_id,
        "transaction_type": tx_type,
        "channel": channel,
        "amount": _amount_around(account_profile[acc_id]),
        "currency": "INR",
        "status": status,
        "transaction_city": city_of_account[acc_id],   # home city
        "counterparty_account": counterparty,
        "description": f"{tx_type} via {channel}",
        "transaction_timestamp": _fmt(_random_timestamp()),
    }


def make_normal_transactions(n):
    # Build n ordinary transactions across random accounts.
    return [make_normal_transaction(rng.choice(account_ids)) for _ in range(n)]

## 14. Generate normal transactions

In [ ]:
normal_rows = make_normal_transactions(N_TRANSACTIONS)
print(f"Generated {len(normal_rows)} normal transactions.")

## 15. Seed deliberate fraud scenarios
We craft KNOWN fraud cases so the Phase 7 rules have something to catch, and we
record each in `ground_truth` so detection accuracy can be measured later.

In [ ]:
fraud_rows = []
ground_truth = []


def _record(row, fraud_type):
    fraud_rows.append(row)
    ground_truth.append({
        "account_id": row["account_id"],
        "transaction_timestamp": row["transaction_timestamp"],
        "amount": row["amount"],
        "fraud_type": fraud_type,
    })


def _base_row(acc_id, amount, ts, channel, status="Success", city=None, card=True):
    return {
        "account_id": acc_id,
        "card_id": account_cards.get(acc_id) if card else None,
        "transaction_type": "Payment",
        "channel": channel,
        "amount": round(amount, 2),
        "currency": "INR",
        "status": status,
        "transaction_city": city or city_of_account[acc_id],
        "counterparty_account": None,
        "description": f"Payment via {channel}",
        "transaction_timestamp": _fmt(ts),
    }


def _distant_city(home):
    return rng.choice([c for c in CITY_NAMES if c != home])


N_EACH = 5   # accounts hit per fraud pattern

# 1) VELOCITY: 6-8 rapid transactions inside a 2-minute window.
for acc_id in rng.sample(account_ids, N_EACH):
    start = _random_timestamp()
    for k in range(rng.randint(6, 8)):
        ts = start + timedelta(seconds=k * 15)   # ~15s apart => all within 2 minutes
        _record(_base_row(acc_id, _amount_around(account_profile[acc_id]), ts, "UPI"), "velocity")

# 2) AMOUNT ANOMALY: one transaction ~20x the account's usual spend.
for acc_id in rng.sample(account_ids, N_EACH):
    big = account_profile[acc_id] * rng.uniform(18, 25)
    _record(_base_row(acc_id, big, _random_timestamp(), "Internet"), "amount_anomaly")

# 3) IMPOSSIBLE TRAVEL: two card transactions minutes apart in distant cities.
for acc_id in rng.sample(carded_accounts, min(N_EACH, len(carded_accounts))):
    home = city_of_account[acc_id]
    t0 = _random_timestamp()
    _record(_base_row(acc_id, _amount_around(account_profile[acc_id]), t0, "POS", city=home),
            "impossible_travel")
    _record(_base_row(acc_id, _amount_around(account_profile[acc_id]), t0 + timedelta(minutes=3),
                      "POS", city=_distant_city(home)), "impossible_travel")

# 4) MIDNIGHT HIGH VALUE: a large transaction between 02:00 and 03:30.
for acc_id in rng.sample(account_ids, N_EACH):
    ts = _random_timestamp().replace(hour=rng.randint(2, 3), minute=rng.randint(0, 30), second=0)
    big = account_profile[acc_id] * rng.uniform(10, 15)
    _record(_base_row(acc_id, big, ts, "Internet"), "midnight_high_value")

# 5) CARD TESTING: several failed attempts seconds apart, then one success.
for acc_id in rng.sample(carded_accounts, min(N_EACH, len(carded_accounts))):
    start = _random_timestamp()
    for k in range(rng.randint(4, 6)):
        ts = start + timedelta(seconds=k * 20)
        _record(_base_row(acc_id, _amount_around(account_profile[acc_id]) * 0.2, ts,
                          "Internet", status="Failed"), "card_testing")
    _record(_base_row(acc_id, _amount_around(account_profile[acc_id]) * 0.2,
                      start + timedelta(seconds=140), "Internet", status="Success"), "card_testing")

print(f"Seeded {len(fraud_rows)} fraudulent transactions across 5 patterns.")

## 16. Combine, sort by time, and save to CSV

In [ ]:
all_rows = normal_rows + fraud_rows
df = pd.DataFrame(all_rows, columns=TX_COLUMNS)
df = df.sort_values("transaction_timestamp").reset_index(drop=True)   # chronological, like a real feed

df.to_csv(TX_CSV_PATH, index=False)

truth_df = pd.DataFrame(ground_truth)
truth_df.to_csv(TRUTH_CSV_PATH, index=False)

print(f"Saved {len(df)} transactions to {TX_CSV_PATH}")
print(f"Saved {len(truth_df)} fraud labels to {TRUTH_CSV_PATH}")

## 17. Quick sanity checks

In [ ]:
print("Total transactions:", len(df))
print("\nStatus breakdown:\n", df["status"].value_counts())
print("\nChannel breakdown:\n", df["channel"].value_counts())
print("\nSeeded fraud types:\n", truth_df["fraud_type"].value_counts())
df.head(10)

## 18. Commit and close

In [ ]:
conn.commit()
cursor.close()
conn.close()
print("Done. Branches, employees, customers, accounts, cards and loans are now in MySQL.")
print("Transactions are in the CSV, ready to be streamed into MySQL in Phase 5.")